# Agent 2 · Trajectoire

Le réseau choisit sept points dans le corridor visible et une vitesse indicative. Le rendu utilise le même plan que le pilote optionnel.

In [ ]:
from dataclasses import dataclass
import cv2
import numpy as np
import torch
from torch import nn

# Coordonnées image normalisées, origine en haut à gauche. De près vers loin.
LIGNES = np.array([.87,.82,.77,.72,.67,.62,.57], np.float32)
N = len(LIGNES)

class GuidageIndisponible(ValueError):
    pass

@dataclass(frozen=True)
class Trajectoire:
    xy: np.ndarray           # N couples (x,y) dans [0,1], près -> loin
    cible_kmh: float

def couloir(probabilite, seuil=.65):
    p = np.asarray(probabilite)
    if p.ndim != 2 or min(p.shape)<32 or not np.isfinite(p).all():
        raise GuidageIndisponible("Carte de piste invalide")
    # Erosion avant le choix des points pour garder une marge aux obstacles.
    libre = cv2.erode((p >= seuil).astype(np.uint8), np.ones((3,3),np.uint8))
    h,w = libre.shape
    if libre.mean() > .92:
        raise GuidageIndisponible("Aucune limite de piste identifiable")
    precedent = None
    limites = []
    for y in np.rint(LIGNES*(h-1)).astype(int):
        transitions = np.diff(np.r_[0, libre[y], 0].astype(int))
        segments = [(a/(w-1),(b-1)/(w-1)) for a,b in
                    zip(np.where(transitions==1)[0],np.where(transitions==-1)[0])
                    if b-a >= max(4,.035*w)]
        if precedent is None:
            segments = [(a,b) for a,b in segments if a <= .5 <= b]
            reference = .5
        else:
            segments = [(a,b) for a,b in segments if min(b,precedent[1]) > max(a,precedent[0])]
            reference = sum(precedent)/2
        if not segments:
            raise GuidageIndisponible("Aucun passage continu devant la voiture")
        precedent = min(segments,key=lambda ab: abs(sum(ab)/2-reference))
        limites.append(precedent)
    return np.asarray(limites,np.float32), libre

class Planificateur(nn.Module):
    def __init__(self):
        super().__init__()
        self.reseau = nn.Sequential(nn.Linear(2*N+2,64),nn.Tanh(),nn.Linear(64,64),
                                    nn.Tanh(),nn.Linear(64,N+1))

    def forward(self, bornes, etat):
        sortie = self.reseau(torch.cat((bornes.flatten(1),etat),dim=1))
        bas,haut = bornes[:,:,0],bornes[:,:,1]
        x = bas + (haut-bas)*(.1 + .8*sortie[:,:N].sigmoid())
        vitesse = 12 + 38*sortie[:,N].sigmoid()
        return x,vitesse

    @torch.inference_mode()
    def proposer(self, probabilite, vitesse_ms, lacet_rads):
        etat = np.asarray([vitesse_ms/30,lacet_rads/3],np.float32)
        if not np.isfinite(etat).all() or vitesse_ms<0:
            raise GuidageIndisponible("Télémétrie invalide")
        bornes, libre = couloir(probabilite)
        x,v = self(torch.from_numpy(bornes)[None],torch.from_numpy(etat)[None])
        points = np.column_stack((x[0].numpy(),LIGNES))
        h,w = libre.shape
        ligne = np.zeros_like(libre)
        pixels = np.rint(points*[w-1,h-1]).astype(np.int32)
        cv2.polylines(ligne,[pixels],False,1,2)
        if np.any((ligne==1) & (libre==0)):
            raise GuidageIndisponible("Trajectoire interrompue par une bordure ou un obstacle")
        return Trajectoire(points,float(v.item()))

def entrainer_planificateur(iterations=600, seed=73):
    # Optimisation sur des corridors variés. Les coûts favorisent un passage
    # fluide, avec marge, et une vitesse prudente quand le passage se resserre.
    torch.manual_seed(seed)
    modele = Planificateur()
    optim = torch.optim.Adam(modele.parameters(),lr=.003)
    t = torch.linspace(0,1,N)[None]
    for _ in range(iterations):
        alea = torch.rand(96,6)
        virage = (alea[:,0:1]-.5)*.70
        centre = .5+(alea[:,1:2]-.5)*.12 + virage*t**2
        rayon = (.30+.15*alea[:,2:3])*(1-.74*t)
        bas,haut = (centre-rayon).clamp(0,1),(centre+rayon).clamp(0,1)
        etat = torch.stack((alea[:,3],alea[:,4]*2-1),dim=1)
        x,v = modele(torch.stack((bas,haut),dim=-1),etat)
        delta = x[:,1:] - x[:,:-1]
        courbure = delta[:,1:]-delta[:,:-1]
        cible = (48-85*virage.abs().flatten()-4*etat[:,1].abs()).clamp(12,48)
        cout = (1.5*(x-centre).square().mean() + 5*(x[:,0]-.5).square().mean()
                + (5+4*etat[:,0:1]).mul(courbure.square()).mean()
                + ((v-cible)/38).square().mean())
        optim.zero_grad()
        cout.backward()
        optim.step()
    return modele.eval()

def dessiner(image, plan=None, raison=""):
    image = cv2.resize(image,(640,480))
    if image.ndim==2:
        image = cv2.cvtColor(image,cv2.COLOR_GRAY2BGR)
    if plan is not None:
        points = np.rint(plan.xy*[639,479]).astype(np.int32)
        cv2.polylines(image,[points],False,(0,240,130),3,cv2.LINE_AA)
        texte = f"Cible indicative  {plan.cible_kmh:.0f} km/h"
    else:
        texte = "Guidage indisponible"
    cv2.putText(image,texte,(12,32),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,255,255),2)
    return image

In [ ]:
def oracle_couloir(virage):
    y,x = np.mgrid[:96,:128]
    t = np.clip((.87-y/95)/.30,0,1)
    centre = .5 + virage*t*t
    largeur = .30-.18*t
    return (abs(x/127-centre)<largeur).astype(np.float32)

def tester_a2(modele):
    plans = [modele.proposer(oracle_couloir(k),8.,0.) for k in (-.23,0.,.23)]
    assert plans[0].xy[-1,0]<.4 and plans[2].xy[-1,0]>.6
    assert max(plans[0].cible_kmh,plans[2].cible_kmh)<plans[1].cible_kmh
    assert np.max(abs(plans[1].xy[:,0]-.5))<.09
    bloque = oracle_couloir(0)
    bloque[66:68,:] = 0  # aussi entre deux lignes de mesure
    for invalide in (np.zeros_like(bloque),np.ones_like(bloque),bloque,np.full_like(bloque,np.nan)):
        try:
            modele.proposer(invalide,8.,0.)
        except GuidageIndisponible:
            continue
        raise AssertionError("Un plan a traversé une zone interdite")
    return {'directions':['gauche','droit','droite'],'cibles_kmh':[p.cible_kmh for p in plans],
            'refus_attendus':4}

In [ ]:
def tester_chaine_humaine(a1,a2):
    rng = np.random.default_rng(564)
    reussis = 0
    franchissements = 0
    for _ in range(40):
        image,masque = scene_synthetique(rng)
        _,p = a1.observer(image)
        try:
            plan = a2.proposer(p,6.,0.)
        except GuidageIndisponible:
            continue
        visuel = dessiner(image,plan)
        assert visuel.shape==(480,640,3)
        h,w = masque.shape
        trace = np.zeros_like(masque)
        cv2.polylines(trace,[np.rint(plan.xy*[w-1,h-1]).astype(np.int32)],False,1,2)
        franchissements += int(np.any((trace>0)&(masque!=1)))
        reussis += 1
    assert reussis>=16, "Couverture insuffisante des scènes synthétiques"
    assert franchissements==0, f"{franchissements} lignes hors du masque de référence"
    return dict(scenes=40,trajectoires_affichees=reussis,franchissements=franchissements)